# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following the Croissant data schema standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --upgrade pip
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load metadata and initialize dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields (schema structure & field identifiers).

We will display all `@id`s for record sets, fields, and columns.

In [ ]:
# List all record sets
print("Available Record Sets (@id):")
for record_set in dataset.schema.record_sets:
    print(f"  - {record_set['@id']} (name: {record_set.get('name','')})")

# For each record set, list fields and their @id
for record_set in dataset.schema.record_sets:
    print(f"\nRecord Set @id: {record_set['@id']} (name: {record_set.get('name','')})")
    print("Fields:")
    for field in record_set.get('field', []):
        if isinstance(field, dict):
            print(f"  - {field['@id']} (name: {field.get('name','')})")
        else:
            # field entry is @id string
            print(f"  - {field}")

In [ ]:
# Preview first few records of each record set using their @id
for record_set in dataset.schema.record_sets:
    record_set_id = record_set['@id']
    print(f"\n--- Records for record set: {record_set_id} ---")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        if i >= 2:
            break
        print(record)

## 3. Data Extraction
Load records from each record set into a pandas DataFrame. Use the `@id` for each record set.

In [ ]:
# Retrieve all record set @id's
record_set_ids = [recset['@id'] for recset in dataset.schema.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Display columns of the primary data table (first populated record set)
if dataframes:
    first_record_set_id = next(iter(dataframes))
    print(f"Columns in DataFrame for '{first_record_set_id}':")
    print(dataframes[first_record_set_id].columns.tolist())
    dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping. All columns and fields are referenced by their `@id` as per Croissant.

We will attempt to identify a numeric field for demonstration.

In [ ]:
# Select a record set and numeric field by their @id
import numpy as np

# List all columns in each DataFrame
for recset_id, df in dataframes.items():
    print(f"\nRecord set: {recset_id}")
    print("Columns:", df.columns.tolist())

# Let's select a numeric field to demonstrate filtering and normalization
# We'll try the first DataFrame with numeric data
selected_record_set_id = None
numeric_field_id = None
for recset_id, df in dataframes.items():
    for col in df.columns:
        # Try to infer if column is numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            selected_record_set_id = recset_id
            numeric_field_id = col
            break
    if numeric_field_id:
        break

if selected_record_set_id and numeric_field_id:
    print(f"\nUsing record set '{selected_record_set_id}' and numeric field '{numeric_field_id}' for EDA.")
    df = dataframes[selected_record_set_id]
    # Set threshold as an arbitrary quantile for demonstration
    threshold = df[numeric_field_id].quantile(0.8) if not df[numeric_field_id].empty else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Attempt grouping by another categorical field
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by '{group_field}':")
        print(grouped_df.head())
    else:
        print("No suitable categorical group field found.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize distribution or relationships in the data. We'll attempt a histogram and, if possible, a boxplot colored by a group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(data=dataframes[selected_record_set_id], x=numeric_field_id, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=dataframes[selected_record_set_id], x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² dataset using the mlcroissant library. We:

- Loaded the schema and reviewed the metadata for context and licensing.
- Inspected available record sets, fields, and their schema `@id`s.
- Loaded data from record sets via their `@id`s.
- Demonstrated filtering, normalization, grouping, and visualization using fields referenced by `@id`.

This workflow enables robust and reproducible data analysis for datasets described with Croissant schemas.

_For more information about this dataset or the Croissant format, please see the [mlcroissant documentation](https://github.com/mlcommons/croissant) or the dataset [source page](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2)._